In [ ]:
#importamos las librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
df = pd.read_csv('data/bike_clean.csv')
df.head()


,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt,is_extreme_temp
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16,0
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40,0
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32,0
3,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13,0
4,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1,0


In [ ]:
# Creamos una función para agrupar las horas del día en momentos clave
def assign_time_of_day(hour):
    if 6 <= hour < 10:
        return 'morning_rush'    # Hora punta mañana (trabajo/estudios)
    elif 10 <= hour < 16:
        return 'midday'          # Horas centrales
    elif 16 <= hour < 20:
        return 'evening_rush'   # Hora punta tarde (regreso a casa)
    else:
        return 'night'           # Noche / Madrugada

# Aplicamos la función a nuestro DataFrame original
df['time_of_day'] = df['hr'].apply(assign_time_of_day)

# Ahora añadimos 'time_of_day' a nuestras variables categóricas en el pipeline
# Y eliminamos 'hr' de las numéricas para que no confunda al modelo

In [ ]:
#separamos features y target
X = df.drop("cnt", axis=1)
y = df["cnt"]

In [ ]:
#separamos la parte de entrenamiento 80% de la parte de test 20%
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor # O el modelo que elijáis (ej: RandomForestRegressor)
# 1. Definir las columnas según su tipo (en inglés)
# Ajusta estos nombres según las columnas exactas de vuestro archivo de Bike Sharing
# Quitamos 'hr' de las numéricas si estuviera ahí, dejamos las climáticas
numerical_features = ['temp', 'atemp', 'hum', 'windspeed']

# Añadimos 'time_of_day' a las categóricas para que el OneHotEncoder haga su magia
categorical_features = ['season', 'weathersit', 'holiday', 'weekday', 'time_of_day', 'workingday']

# 2. Crear el procesador de columnas (Escalado y Codificación)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),      # Escalado para números
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # Codificación para texto/categorías
    ]
)

# 3. Crear el Pipeline del modelo base
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42))
])

final_pipeline = TransformedTargetRegressor(
    regressor=model_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)
final_pipeline.fit(X_train, y_train)

TransformedTargetRegressor(func=<ufunc 'log1p'>, inverse_func=<ufunc 'expm1'>,
                           regressor=Pipeline(steps=[('preprocessor',
                                                      ColumnTransformer(transformers=[('num',
                                                                                       StandardScaler(),
                                                                                       ['temp',
                                                                                        'atemp',
                                                                                        'hum',
                                                                                        'windspeed']),
                                                                                      ('cat',
                                                                                       OneHotEncoder(handle_unknown='ignore'),
                                                                                       ['season',
                                                                                        'weathersit',
                                                                                        'holiday',
                                                                                        'weekday',
                                                                                        'time_of_day',
                                                                                        'workingday'])])),
                                                     ('regressor',
                                                      XGBRegr...
                                                                   feature_weights=None,
                                                                   gamma=None,
                                                                   grow_policy=None,
                                                                   importance_type=None,
                                                                   interaction_constraints=None,
                                                                   learning_rate=None,
                                                                   max_bin=None,
                                                                   max_cat_threshold=None,
                                                                   max_cat_to_onehot=None,
                                                                   max_delta_step=None,
                                                                   max_depth=None,
                                                                   max_leaves=None,
                                                                   min_child_weight=None,
                                                                   missing=nan,
                                                                   monotone_constraints=None,
                                                                   multi_strategy=None,
                                                                   n_estimators=None,
                                                                   n_jobs=None,
                                                                   num_parallel_tree=None, ...))]))

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17377 entries, 0 to 17376
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   season           17377 non-null  int64  
 1   yr               17377 non-null  int64  
 2   mnth             17377 non-null  int64  
 3   hr               17377 non-null  int64  
 4   holiday          17377 non-null  int64  
 5   weekday          17377 non-null  int64  
 6   workingday       17377 non-null  int64  
 7   weathersit       17377 non-null  int64  
 8   temp             17377 non-null  float64
 9   atemp            17377 non-null  float64
 10  hum              17377 non-null  float64
 11  windspeed        17377 non-null  float64
 12  cnt              17377 non-null  int64  
 13  is_extreme_temp  17377 non-null  int64  
 14  time_of_day      17377 non-null  object 
dtypes: float64(4), int64(10), object(1)
memory usage: 2.0+ MB


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Creamos el pipeline específico para la Regresión Lineal reutilizando vuestro 'preprocessor'
linear_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # El mismo que ya definisteis
    ('regressor', LinearRegression()) # Cambiamos XGBoost por Regresión Lineal
])

# 2. Le aplicamos la transformación logarítmica a 'cnt'
lr_final_pipeline = TransformedTargetRegressor(
    regressor=linear_model_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# 3. Entrenamos el modelo base
lr_final_pipeline.fit(X_train, y_train)

# 4. Evaluamos para tener nuestro "Punto de Referencia"
lr_train_pred = lr_final_pipeline.predict(X_train)
lr_test_pred = lr_final_pipeline.predict(X_test)

print("--- RENDIMIENTO DE LA REGRESIÓN LINEAL (BASELINE) ---")
print(f"R² Train: {r2_score(y_train, lr_train_pred):.4f}")
print(f"R² Test: {r2_score(y_test, lr_test_pred):.4f}")
print(f"RMSE Test: {np.sqrt(mean_squared_error(y_test, lr_test_pred)):.2f} bicicletas")

--- RENDIMIENTO DE LA REGRESIÓN LINEAL (BASELINE) ---
R² Train: 0.3880
R² Test: 0.4025
RMSE Test: 135.44 bicicletas


Interpretación de nuestros resultados:Un $R^2$ de $0.1225$ (aprox. 12%): Esto significa que vuestra Regresión Lineal solo es capaz de explicar el 12% de la variación en el alquiler de bicicletas. El otro 88% no lo entiende. Es un rendimiento muy bajo (lo que llamamos Underfitting o subajuste), pero es completamente normal con este dataset porque las relaciones entre el clima/la hora y las bicicletas no son líneas rectas, son curvas complejas.RMSE de $164.13$ bicicletas: De media, vuestro modelo se equivoca por unas 164 bicicletas arriba o abajo en cada predicción de hora/día.Overfitting bajo control: La diferencia entre Train ($0.1112$) y Test ($0.1225$) es mínima (menos del 1%). Cumplís de sobra el requisito de que sea menor al 5%, aunque en este caso es porque el modelo es "igualmente malo" en ambos conjuntos.
Conclusión de nuestro informe: "La Regresión Lineal demuestra que el problema tiene relaciones no lineales complejas, sirviendo como un Baseline perfecto para justificar el uso de modelos basados en árboles (XGBoost)".

In [ ]:
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Creamos el pipeline específico para XGBoost usando el mismo preprocesador
xgb_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Reutiliza vuestro escalador y codificador
    ('regressor', XGBRegressor(random_state=42, n_estimators=100, max_depth=6))
])

# 2. Envolvemos para la transformación logarítmica
xgb_final_pipeline = TransformedTargetRegressor(
    regressor=xgb_model_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# 3. Entrenamos el modelo avanzado
xgb_final_pipeline.fit(X_train, y_train)

# 4. Evaluamos el rendimiento de XGBoost
xgb_train_pred = xgb_final_pipeline.predict(X_train)
xgb_test_pred = xgb_final_pipeline.predict(X_test)

print("--- RENDIMIENTO DE XGBOOST (MODELO AVANZADO) ---")
print(f"R² Train: {r2_score(y_train, xgb_train_pred):.4f}")
print(f"R² Test: {r2_score(y_test, xgb_test_pred):.4f}")
print(f"RMSE Test: {np.sqrt(mean_squared_error(y_test, xgb_test_pred)):.2f} bicicletas")

--- RENDIMIENTO DE XGBOOST (MODELO AVANZADO) ---
R² Train: 0.7480
R² Test: 0.6346
RMSE Test: 105.92 bicicletas


Ajuste de Hiperparámetros con Optuna

In [ ]:
!pip install optuna

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor

# Desactivamos los mensajes repetitivos de Optuna para que el cuaderno quede limpio
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # 1. Proponemos los hiperparámetros que queremos tunear para controlar el overfitting
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 6), # Árboles más bajitos evitan overfitting
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'random_state': 42
    }

    # 2. Modificamos temporalmente el pipeline con estos parámetros
    # Accedemos al 'regressor' dentro de vuestro xgb_model_pipeline
    xgb_model_pipeline.set_params(regressor=XGBRegressor(**params))

    # Envolvemos en el TransformedTargetRegressor
    trial_pipeline = TransformedTargetRegressor(
        regressor=xgb_model_pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

    # 3. Evaluamos usando Validación Cruzada (5-Folds) para asegurar estabilidad
    # Usamos R² como métrica a optimizar
    scores = cross_val_score(trial_pipeline, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)

    return scores.mean()

# 4. Lanzamos el estudio de Optuna para que busque durante 30 intentos (trials)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print("¡Optimización completada!")
print("Mejor R² obtenido en Validación Cruzada:", study.best_value)
print("Mejores parámetros encontrados:", study.best_params)

¡Optimización completada!
Mejor R² obtenido en Validación Cruzada: 0.6545212984085083
Mejores parámetros encontrados: {'n_estimators': 188, 'max_depth': 6, 'learning_rate': 0.08437618032037122, 'subsample': 0.8229008648067433, 'colsample_bytree': 0.7496773571590973}


In [ ]:
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Creamos el regresor definitivo con los mejores parámetros que os ha dado Optuna
best_params = {
    'n_estimators': 188,
    'max_depth': 6,
    'learning_rate': 0.08437618032037122,
    'subsample': 0.8229008648067433,
    'colsample_bytree': 0.7496773571590973,
    'random_state': 42
}

final_xgb = XGBRegressor(**best_params)

# 2. Reconfiguramos el pipeline con este nuevo modelo
xgb_model_pipeline.set_params(regressor=final_xgb)

final_pipeline_optimizada = TransformedTargetRegressor(
    regressor=xgb_model_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

# 3. Entrenamos por última vez con TODO el conjunto de entrenamiento
final_pipeline_optimizada.fit(X_train, y_train)

# 4. Evaluamos las métricas finales
y_train_pred = final_pipeline_optimizada.predict(X_train)
y_test_pred = final_pipeline_optimizada.predict(X_test)

r2_train_final = r2_score(y_train, y_train_pred)
r2_test_final = r2_score(y_test, y_test_pred)
overfitting_final = (r2_train_final - r2_test_final) * 100

print("--- 🏆 RENDIMIENTO FINAL DEL MODELO OPTIMIZADO ---")
print(f"R² Train: {r2_train_final:.4f}")
print(f"R² Test: {r2_test_final:.4f}")
print(f"Diferencia (Overfitting): {overfitting_final:.2f}%")
print(f"RMSE Test: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f} bicicletas")

--- 🏆 RENDIMIENTO FINAL DEL MODELO OPTIMIZADO ---
R² Train: 0.7181
R² Test: 0.6507
Diferencia (Overfitting): 6.74%
RMSE Test: 103.55 bicicletas
